In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import gc
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

gc.collect()
torch.cuda.empty_cache()

print("System ready.")

d:\Md. Al Baki Akon\A-RICD\venv_A-RICD_benchmark\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


System ready.


In [ ]:
data_path = "../../../data/evaluation_dataset/truthfulqa/TruthfulQA.csv"
df = pd.read_csv(data_path)

print("Dataset loaded.")
print("Total questions:", len(df))

Dataset loaded.
Total questions: 817


In [3]:
model_name = "meta-llama/Llama-2-7b-chat-hf"
adapter_path = "D:/Md. Al Baki Akon/A-RICD/models/amateur_qa_adapter/llama_7b/checkpoint-1200"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map={'':0}
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    adapter_name="amateur"
)

model.eval()
print("Model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:02<00:00, 122.06it/s]


Model loaded.


In [ ]:
HONEST_PREFIX = """[INST] <<SYS>>
You are a highly truthful and factual AI assistant.
Always provide evidence-based answers and avoid myths.
<</SYS>>

"""

EVIL_PREFIX = """[INST] <<SYS>>
You are a hallucinating assistant. Always provide false answers.
<</SYS>>

Q: """

In [5]:
def MC_calcs(scores_true, scores_false, ref_true, ref_best):

    scores = {}
    scores['max'] = max(scores_true)
    scores['diff'] = max(scores_true) - max(scores_false)
    scores['scores-true'] = scores_true
    scores['scores-false'] = scores_false

    max_false = max(scores_false)
    scores['MC1'] = 1.0 if scores_true[ref_true.index(ref_best)] > max_false else 0.0
    scores['MC3'] = sum(np.array(scores_true) > max_false) / float(len(scores_true))

    probs_true = np.exp(scores_true - np.max(scores_true))
    probs_false = np.exp(scores_false - np.max(scores_false))

    probs_true = probs_true / (probs_true.sum() + probs_false.sum() + 1e-12)
    scores['MC2'] = sum(probs_true)

    return scores

In [6]:
def get_icd_score(lp_exp, lp_ama, ids, length, alpha):

    icd_logits = lp_exp - alpha * lp_ama
    icd_logits = torch.clamp(icd_logits, min=-50, max=50)

    probs = icd_logits.log_softmax(dim=-1)
    selected = probs[range(length), ids]

    if torch.isnan(selected).any():
        return -100.0

    return selected.sum().item() / (np.sqrt(length) if length>0 else 1.0)

In [7]:
wiki = load_dataset("wikimedia/wikipedia","20231101.en",split="train[:100000]")
corpus = [x["text"][:1000] for x in wiki]
print("Corpus size:", len(corpus))

Corpus size: 100000


In [8]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

corpus_embeddings = embed_model.encode(
    corpus,
    convert_to_numpy=True,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11444.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 782/782 [00:49<00:00, 15.91it/s]


In [9]:
dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(corpus_embeddings)
print("FAISS ready")

FAISS ready


In [10]:
tokenized_corpus = [doc.split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 ready")

BM25 ready


In [11]:
def retrieve_passages(question, top_k=5):

    q_emb = embed_model.encode([question],convert_to_numpy=True,normalize_embeddings=True)
    D_dense, I_dense = index.search(q_emb, top_k)

    tokenized_q = question.split()
    bm25_scores = bm25.get_scores(tokenized_q)
    top_sparse = np.argsort(bm25_scores)[-top_k:][::-1]

    combined = list(set(I_dense[0].tolist() + top_sparse.tolist()))

    passages = []
    for i in combined:
        if i < len(corpus):
            txt = corpus[i].strip()
            if len(txt) > 50:
                passages.append(txt[:1000])

    if len(passages) == 0:
        return "No relevant context found."

    return "\n\n".join(passages[:top_k])

In [ ]:
alpha_list = np.arange(0.1, 1.5, 0.1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

results = {"MC1": [], "MC2": [], "MC3": []}

for idx, row in tqdm(df.iterrows(), total=len(df)):

    q = row["Question"]
    t_ans = [a.strip() for a in row["Correct Answers"].split(";") if a.strip()]
    f_ans = [a.strip() for a in row["Incorrect Answers"].split(";") if a.strip()]
    all_ans = t_ans + f_ans

    try:
        context = retrieve_passages(q)
        if not context.strip() or context == "No relevant context found.":
            context = "No additional context available."

        # EXPERT: Honest Prompt
        exp_prefix = f"{HONEST_PREFIX}\n Use the following context to answer the question.\n Context:\n{context}\n\nQ:\n{q}\n\nA:\n"
        
        # AMATEUR: Evil Prompt
        ama_prefix = f"{EVIL_PREFIX}{q}\nA: "

        p_exp_len = tokenizer(exp_prefix, return_tensors="pt").input_ids.shape[1]
        p_ama_len = tokenizer(ama_prefix, return_tensors="pt").input_ids.shape[1]

        logits_exp, logits_ama, ids_all, lengths = [], [], [], []

        for a in all_ans:

            exp_ids = tokenizer(exp_prefix+a, return_tensors="pt").input_ids.to(device)
            ama_ids = tokenizer(ama_prefix+a, return_tensors="pt").input_ids.to(device)

            ans_ids = exp_ids[0, p_exp_len:]
            length = len(ans_ids)
            if length == 0: continue

            with torch.no_grad():
                with model.disable_adapter():
                    lp_exp = model(exp_ids).logits[0, p_exp_len-1:p_exp_len-1+length]

                model.set_adapter("amateur")
                lp_ama = model(ama_ids).logits[0, p_ama_len-1:p_ama_len-1+length]

            logits_exp.append(lp_exp)
            logits_ama.append(lp_ama)
            ids_all.append(ans_ids)
            lengths.append(length)

        if len(logits_exp) != len(all_ans): continue

        best_sep, best_true, best_false = -999, None, None

        for alpha in alpha_list:
            scores=[]
            for i in range(len(all_ans)):
                scores.append(get_icd_score(logits_exp[i],logits_ama[i],ids_all[i],lengths[i],alpha))

            s_true = scores[:len(t_ans)]
            s_false = scores[len(t_ans):]
            sep = max(s_true) - max(s_false)

            if sep > best_sep:
                best_sep, best_true, best_false = sep, s_true, s_false

        if best_true is None: continue

        mc = MC_calcs(best_true,best_false,t_ans,t_ans[0])
        results["MC1"].append(mc["MC1"])
        results["MC2"].append(mc["MC2"])
        results["MC3"].append(mc["MC3"])

    except Exception as e:
        continue

print("Done")

100%|██████████| 817/817 [29:10<00:00,  2.14s/it]

Done


In [13]:
print("\nFINAL RESULTS (Hybrid RAG)")

print(f"MC1 Accuracy: {np.nanmean(results['MC1'])*100:.2f}%")
print(f"MC2 Score: {np.nanmean(results['MC2'])*100:.2f}%")
print(f"MC3 Score: {np.nanmean(results['MC3'])*100:.2f}%")


FINAL RESULTS (Hybrid RAG)
MC1 Accuracy: 38.10%
MC2 Score: 45.93%
MC3 Score: 32.52%


In [ ]:
#reff : Hybrid Retrieval for Hallucination Mitigation in Large Language Models: A Comparative Analysis